[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sensioai/blog/blob/master/028_pytorch_nn/pytorch_nn.ipynb)

# Pytorch - Redes Neuronales

se importa la librería torch para su manejo "tensores"

In [51]:
import torch

## Modelos secuenciales

La forma más sencilla de definir una `red neuronal` en `Pytorch` es utilizando la clase `Sequentail`. Esta clase nos permite definir una secuencia de capas, que se aplicarán de manera secuencial. Ésto ya lo conocemos de posts anteriores, ya que es la forma ideal de definir un `Perceptrón Multicapa`

In [52]:
D_in, H, D_out = 7, 100, 3

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
)

In [53]:
outputs = model(torch.randn(600, 7))
outputs.shape

torch.Size([600, 3])

In [54]:
print(outputs[0][:])

tensor([ 0.3222, -0.0736,  0.2724], grad_fn=<SliceBackward0>)


In [55]:
model

Sequential(
  (0): Linear(in_features=7, out_features=100, bias=True)
  (1): ReLU()
  (2): Linear(in_features=100, out_features=3, bias=True)
)

In [56]:
model.to("cuda")

Sequential(
  (0): Linear(in_features=7, out_features=100, bias=True)
  (1): ReLU()
  (2): Linear(in_features=100, out_features=3, bias=True)
)

#### Cargamos los datos, desde nuestro csv en data para poder entrenar a nuestro modelo

In [57]:
from pandas import read_csv

data = read_csv("Seed_data.csv")
X, Y = torch.tensor(data.iloc[:, :-1].values).float(), torch.tensor(data.iloc[:, -1].values).long()

X.shape, Y.shape

(torch.Size([209, 7]), torch.Size([209]))

#### Normalizamos los datos para evitar errores y desbordamiento de datos además separamos en train 90% y test 10% 

In [58]:
import numpy as np

x_2 = np.array(X)
y_2 = np.array(Y)

# Normalización y split
X_train = x_2[:189]   # primeros 189 para entrenamiento (90%)
X_test  = x_2[189:]   # últimos 21 para prueba (10%)

# Normalización 0-1
X_train = (X_train - 1) / 20
X_test  = (X_test - 1) / 20

# Convertir etiquetas a enteros
y_train = y_2[:189].astype(np.int32)
y_test  = y_2[189:].astype(np.int32)


#### softmax convierte los valores de salida de la red en probabilidades que suman 1, cross_entropy mide qué tan lejos están esas predicciones de las clases correctas y nos da un número para minimizar durante el entrenamiento.

In [59]:
# función de pérdida y derivada

def softmax(x):
    return torch.exp(x) / torch.exp(x).sum(axis=-1,keepdims=True)

def cross_entropy(output, target):
    logits = output[torch.arange(len(output)), target]
    loss = - logits + torch.log(torch.sum(torch.exp(output), axis=-1))
    loss = loss.mean()
    return loss

In [60]:
X_train

array([[ 0.694     ,  0.6785    , -0.005945  , ...,  0.11665   ,
         0.0009    ,  0.1978    ],
       [ 0.6645    ,  0.6545    , -0.00475   , ...,  0.11685   ,
         0.08494999,  0.19125   ],
       [ 0.642     ,  0.64699996, -0.005225  , ...,  0.11894999,
         0.06295   ,  0.19025   ],
       ...,
       [ 0.4955    ,  0.59000003, -0.00814   , ...,  0.08374999,
         0.15895   ,  0.1978    ],
       [ 0.5115    ,  0.59099996, -0.00703   , ...,  0.09105001,
         0.3262    ,  0.19784999],
       [ 0.4795    ,  0.5705    , -0.00676   , ...,  0.08935   ,
         0.19874999,  0.1897    ]], dtype=float32)

#### verificamos si se tiene activa la GPU para usarlo en el entrenamiento del modelo

In [61]:
torch.cuda.is_available()

True

In [62]:
print(X)

tensor([[14.8800, 14.5700,  0.8811,  ...,  3.3330,  1.0180,  4.9560],
        [14.2900, 14.0900,  0.9050,  ...,  3.3370,  2.6990,  4.8250],
        [13.8400, 13.9400,  0.8955,  ...,  3.3790,  2.2590,  4.8050],
        ...,
        [13.2000, 13.6600,  0.8883,  ...,  3.2320,  8.3150,  5.0560],
        [11.8400, 13.2100,  0.8521,  ...,  2.8360,  3.5980,  5.0440],
        [12.3000, 13.3400,  0.8684,  ...,  2.9740,  5.6370,  5.0630]])


#### Convertimos los datos de entrenamiento a tensores y los enviamos a la GPU, en cada época hacemos forward con el modelo, calculamos la pérdida cross_entropy y registramos su valor, posteriormente aplicamos backprop para calcular gradientes y actualizamos los pesos con el learning rate, repitiendo esto durante 500 épocas.

In [63]:
# convertimos datos a tensores y copiamos en gpu

X_t = torch.from_numpy(X_train).float().cuda()
Y_t = torch.from_numpy(y_train).long().cuda()

# bucle entrenamiento
epochs = 500
lr = 0.8
log_each = 10
l = []
for e in range(1, epochs + 1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = cross_entropy(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    model.zero_grad()

    # Backprop (calculamos todos los gradientes automáticamente)
    loss.backward()

    # update de los pesos
    with torch.no_grad():
        for param in model.parameters():
            param -= lr * param.grad

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

Epoch 10/500 Loss 1.02080
Epoch 20/500 Loss 0.95436
Epoch 30/500 Loss 0.96092
Epoch 40/500 Loss 0.93921
Epoch 50/500 Loss 0.91485
Epoch 60/500 Loss 0.89038
Epoch 70/500 Loss 0.86638
Epoch 80/500 Loss 0.84307
Epoch 90/500 Loss 0.82022
Epoch 100/500 Loss 0.79627
Epoch 110/500 Loss 0.78844
Epoch 120/500 Loss 0.77052
Epoch 130/500 Loss 0.74595
Epoch 140/500 Loss 0.74003
Epoch 150/500 Loss 0.72730
Epoch 160/500 Loss 0.71161
Epoch 170/500 Loss 0.69496
Epoch 180/500 Loss 0.68479
Epoch 190/500 Loss 0.66727
Epoch 200/500 Loss 0.65912
Epoch 210/500 Loss 0.64484
Epoch 220/500 Loss 0.63375
Epoch 230/500 Loss 0.62317
Epoch 240/500 Loss 0.61075
Epoch 250/500 Loss 0.59652
Epoch 260/500 Loss 0.58285
Epoch 270/500 Loss 0.57171
Epoch 280/500 Loss 0.56954
Epoch 290/500 Loss 0.55777
Epoch 300/500 Loss 0.54753
Epoch 310/500 Loss 0.54071
Epoch 320/500 Loss 0.53252
Epoch 330/500 Loss 0.52230
Epoch 340/500 Loss 0.51257
Epoch 350/500 Loss 0.50349
Epoch 360/500 Loss 0.49550
Epoch 370/500 Loss 0.48937
Epoch 380/

#### Vemos la precisión de nuestro modelo, en base  nustro y_test y y_pred, nos devuelve una probabilidad, de nuestra precisión

In [64]:
from sklearn.metrics import accuracy_score

def evaluate(x):
    model.eval()
    y_pred = model(x)
    y_probas = softmax(y_pred)
    return torch.argmax(y_probas, axis=1)

y_pred = evaluate(torch.from_numpy(X_test).float().cuda())
accuracy_score(y_test, y_pred.cpu().numpy())

0.9

#### como nos sale 0.9, entonces la precisión del modelo es del 90%!!!